# Homework 8: Neural Networks and Deep Learning

Classify curly and straight hair with the fixed 2026 PyTorch CPU setup. This notebook validates the dataset and model, reuses the reference trainer only when histories are missing, and calculates the six submission answers.

## Reproducible setup

Use Python 3.11 with the versions pinned in `requirements-cpu.txt`. The expected dataset archive SHA-256 is `9e53453e3017502f22860cb08558f0cdb343e102fb61feab75960616185e7d67`. The archive extracts under `data/data/` in this workspace.

In [ ]:
import hashlib
import json
import random
import statistics
import subprocess
import sys
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torchvision
from torchvision.datasets import ImageFolder
from torchvision import transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.set_num_threads(2)
DEVICE = torch.device("cpu")
NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "reference_train.py").exists():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "cohorts" / "2026" / "homework" / "08-deep-learning"
DATA_DIR = NOTEBOOK_DIR / "data"
OUTPUT_DIR = NOTEBOOK_DIR

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("torchvision:", torchvision.__version__)

In [ ]:
archive = NOTEBOOK_DIR / "data.zip"
if archive.exists():
    digest = hashlib.sha256(archive.read_bytes()).hexdigest()
    print("data.zip SHA-256:", digest)
    assert digest == "9e53453e3017502f22860cb08558f0cdb343e102fb61feab75960616185e7d67"

train_dataset = ImageFolder(DATA_DIR / "train")
evaluation_dataset = ImageFolder(DATA_DIR / "test")
assert len(train_dataset) == 800
assert len(evaluation_dataset) == 201
assert train_dataset.class_to_idx == {"curly": 0, "straight": 1}
print("Train examples:", len(train_dataset))
print("Evaluation examples:", len(evaluation_dataset))
print("Class mapping:", train_dataset.class_to_idx)

## Model and loss

The network returns one logit per image. `BCEWithLogitsLoss` combines sigmoid and binary cross-entropy stably; do not add a sigmoid layer to the model.

In [ ]:
class HairModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=0)
        self.pool = nn.MaxPool2d(kernel_size=2)
        self.hidden = nn.Linear(32 * 99 * 99, 64)
        self.output = nn.Linear(64, 1)

    def forward(self, x):
        x = self.pool(nn.functional.relu(self.conv(x)))
        x = x.flatten(start_dim=1)
        x = nn.functional.relu(self.hidden(x))
        return self.output(x)

model = HairModel().to(DEVICE)
parameter_count = sum(parameter.numel() for parameter in model.parameters())
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.002, momentum=0.8)
print("Trainable parameters:", parameter_count)
print("Loss:", criterion)
assert parameter_count == 20073473

## Training histories

The reference trainer applies the specified baseline phase followed by 10 augmented epochs without recreating the model or optimizer. This cell skips training if both history files already exist.

In [ ]:
baseline_path = OUTPUT_DIR / "history_baseline.json"
augmented_path = OUTPUT_DIR / "history_augmented.json"
if not (baseline_path.exists() and augmented_path.exists()):
    subprocess.run(
        [
            sys.executable,
            str(NOTEBOOK_DIR / "reference_train.py"),
            "--data-dir",
            str(DATA_DIR),
            "--output-dir",
            str(OUTPUT_DIR),
        ],
        check=True,
    )
else:
    print("Using existing histories in", OUTPUT_DIR)

In [ ]:
baseline = json.loads(baseline_path.read_text(encoding="utf-8"))
augmented = json.loads(augmented_path.read_text(encoding="utf-8"))
expected_keys = {"train_loss", "train_accuracy", "evaluation_loss", "evaluation_accuracy"}
assert set(baseline) == expected_keys
assert set(augmented) == expected_keys
assert all(len(values) == 10 for values in baseline.values())
assert all(len(values) == 10 for values in augmented.values())

baseline_median_accuracy = statistics.median(baseline["train_accuracy"])
baseline_loss_std = float(np.std(baseline["train_loss"], ddof=0))
augmented_mean_loss = statistics.mean(augmented["evaluation_loss"])
augmented_last_five_accuracy = statistics.mean(augmented["evaluation_accuracy"][-5:])

print("Baseline median train accuracy:", baseline_median_accuracy)
print("Baseline train-loss population std:", baseline_loss_std)
print("Augmented mean evaluation loss:", augmented_mean_loss)
print("Augmented mean evaluation accuracy, last 5 epochs:", augmented_last_five_accuracy)

In [ ]:
answers = {
    1: "nn.BCEWithLogitsLoss()",
    2: parameter_count,
    3: round(baseline_median_accuracy, 2),
    4: round(baseline_loss_std, 3),
    5: round(augmented_mean_loss, 3),
    6: round(augmented_last_five_accuracy, 2),
}
for question, answer in answers.items():
    print(f"{question}. {answer}")